In [ ]:
%load_ext autoreload
%autoreload 2
import os
import pandas as pd
import os
import numpy as np
import scprinter as scp
from tqdm.auto import *

In [ ]:
work_dir = '/ewsc/zhangruo/RSav_scprinter'

In [ ]:
meta = pd.read_csv(os.path.join(work_dir, 'meta.dedoublet.csv'))
meta

In [ ]:
import scprinter as scp
import time
start = time.time()

if os.path.exists(f'{work_dir}/scprinter.h5ad'):
    printer = scp.load_printer(f'{work_dir}/scprinter.h5ad', scp.genome.hg38)
else:
    printer = scp.pp.import_fragments(
                        path_to_frags=os.path.join(work_dir, 'All.frag.tsv.gz'),
                        barcodes=list(meta['barcode']),
                        savename=f'{work_dir}/scprinter.h5ad',
                        genome=scp.genome.hg38,
                        min_num_fragments=0, min_tsse=0,
                        sorted_by_barcode=False, 
                        low_memory=False
                        )
print ("takes", time.time()-start)

In [ ]:
printer

In [ ]:
printer.close()

In [ ]:
# Use all barcodes presented, and call peaks
scp.pp.call_peaks(printer=printer, 
                  frag_file=os.path.join(work_dir, 'All.frag.tsv.gz'),
                  cell_grouping=[list(meta['barcode'])
                                 ], # here we call peaks on the cells that are included in the final analyses
                  group_names=['Bulk'], 
                  preset='seq2PRINT',
                  iterative_peak_merging=True,
                  merge_across_groups=False,
                 )

In [ ]:
cleaned_peaks = pd.DataFrame(printer.uns['peak_calling'][f'Bulk_cleaned'][:])
cleaned_peaks.to_csv(f'{work_dir}/seq2print_Bulk_cleaned_narrowPeak.bed', 
                 sep='\t', header=False, index=False)

In [ ]:
import json
model_configs = []
if not os.path.exists(os.path.join(work_dir, 'configs')):
    os.makedirs(os.path.join(work_dir, 'configs'))

for fold in range(5):
    model_config = scp.tl.seq_model_config(printer, 
                                     region_path=f'{work_dir}/seq2print_Bulk_cleaned_narrowPeak.bed',
                                     cell_grouping=[list(printer.obs_names)],
                                     group_names=['Bulk'],
                                     genome=printer.genome,
                                     fold=fold,
                                     overwrite_bigwig=False,
                                     model_name="Bulk",
                                     additional_config={
                                        "notes": "v3",
                                        "tags": ["MORF",
                                            "Bulk",
                                            f"fold{fold}"]},
                                     path_swap=(work_dir, ''),
                                     config_save_path=f'{work_dir}/configs/Bulk_fold{fold}.JSON')
    model_configs.append(model_config)

In [ ]:
for path in ['temp','model']:
    if not os.path.exists(os.path.join(work_dir, path)):
        os.makedirs(os.path.join(work_dir, path))
        
name = 'Bulk'
for fold in range(5):
    scp.tl.launch_seq2print(model_config_path=f'{work_dir}/configs/{name}_fold{fold}.JSON',
                            temp_dir=f'{work_dir}/temp',
                            model_dir=f'{work_dir}/model',
                            data_dir=f'{work_dir}',
                            gpus=fold,
                            wandb_project='scPrinter_seq_RSav_morf', # wandb helps you manage loggins
                            verbose=False, 
                            launch=False # launch=True, this command would launch the scripts directly, 
                            # otherwise, it will just display the commands, you should copy them and run them.
                           )

In [ ]:
# Now fetch models:

import wandb

# Login to your W&B account
wandb.login()

# Set your entity and project
entity = 'ruochiz'  # Replace with your W&B entity (username or team name)
project = 'scPrinter_seq_RSav_morf'  # Replace with your W&B project name

# Initialize the API
api = wandb.Api()

# Get the project
runs = api.runs(f"{entity}/{project}")
condition2model = {}
for run in runs:
    model_name = run.config['savename'] + '-' + run.name + '.pt'
    if run.state == 'finished':
        for tag in run.tags:
            if 'MORF' in tag:
                continue
            if 'fold' in tag:
                continue
            condition = tag
            if condition not in condition2model:
                condition2model[condition] = []
        condition2model[condition].append(model_name)

In [ ]:
condition2model = {'Bulk': condition2model['Bulk']}

In [ ]:
condition2model

In [ ]:
peak = os.path.join(work_dir, 'cleanpeaks.bed')

In [ ]:
for condition in condition2model:
    model_path = [f'{work_dir}/model/{m}' for m in condition2model[condition]]
    # This will generate the sequence attribution scores for the footprint, you can input multiple models, it will iterate over them
    scp.tl.seq_attr_seq2print(
        genome=printer.genome,
        region_path=peak,
        model_type='seq2print',
        model_path=model_path,
        gpus=[0,1,2,3,4,5,6,7],
        preset='footprint',
        overwrite=False,
        verbose=False,
        launch=False)
    scp.tl.seq_attr_seq2print(
        genome=printer.genome,
        region_path=peak,
        model_type='seq2print',
        model_path=model_path,
        gpus=[0,1,2,3,4,5,6,7],
        preset='count',
        overwrite=False,
        verbose=False,
        launch=False)

In [ ]:
if not os.path.exists(f'{work_dir}/modisco'):
    os.makedirs(f'{work_dir}/modisco')
for condition in condition2model:
    model_path = [f'{work_dir}/model/{m}' for m in condition2model[condition]]
    scp.tl.seq_denovo_seq2print(model_path=model_path,
                                region_path=peak,
                                genome=printer.genome,
                                gpus=[0,1,2,3,4,5,6,7],
                                preset='footprint',
                                save_path=f'{work_dir}/modisco/modisco.footprint.{condition}.h5',
                                verbose=False, overwrite=False, launch=False)
    scp.tl.seq_denovo_seq2print(model_path=model_path,
                                region_path=peak,
                                genome=printer.genome,
                                gpus=[0,1,2,3,4,5,6,7],
                                preset='count',
                                save_path=f'{work_dir}/modisco/modisco.count.{condition}.h5',
                                verbose=False, overwrite=False, launch=False)

In [ ]:
for extra_str in ['', '_rb_5', '_rb_10', '_rb_20']:
    modisco_h5 = []
    delta_effect_path = []
    motif_prefix = []
    for condition in condition2model:
        for kind in ['footprint', 'count']:
            modisco_h5.append(f'{work_dir}/modisco/modisco.{kind}.{condition}{extra_str}.h5')
            delta_effect_path.append(f'{work_dir}/modisco/{kind}.{condition}{extra_str}')
            motif_prefix.append(f'{kind}.{condition}.')
            
    df = scp.tl.modisco_report(modisco_h5=modisco_h5,
                         save_path=f'{work_dir}/modisco_report_all{extra_str}',
                         meme_motif=scp.datasets.FigR_motifs_human_meme,
                         top_n_matches=3,
                         delta_effect_path=delta_effect_path,
                         motif_prefix=motif_prefix)
            

In [ ]:
peak = os.path.join(work_dir, 'cleanpeaks.bed')
peak_df = scp.utils.regionparser(peak)
peak_df

In [ ]:
hits_all = []
names_all = []
for condition in condition2model:
    model_path = [f'{work_dir}/model/{m}' for m in condition2model[condition]]
    for extra_str in ['', '_rb_5', '_rb_10', '_rb_20']:
        for kind in ['footprint', 'count']:
            scp.tl.seq_denovo_callhits(modisco_output=f'{work_dir}/modisco/modisco.{kind}.{condition}{extra_str}.h5',
                                       model_path=model_path,
                                       region_path=peak,
                                       device='cuda:0',
                                       preset=kind,
                                       save_path=f'{work_dir}/modisco/{kind}.{condition}{extra_str}/finemo',
                                       overwrite=False,
                                       verbose=True,
                                       launch=True, # Usually should be True when return_hits=True, but I run it once already
                                       return_hits=False)
           

In [ ]:
adata_atac = scp.pp.make_peak_matrix(printer, peak)

In [ ]:
adata_atac_backup = adata_atac.copy()

In [ ]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="6"
import warnings
warnings.filterwarnings("ignore")
import cupy as cp
os.environ['RAY_TQDM_PATCH_PRINT'] = '0'
import rmm
from rmm.allocators.cupy import rmm_cupy_allocator
rmm.reinitialize(
    managed_memory=True, # Allows oversubscription
    pool_allocator=True, # default is False
    devices=0, # GPU device IDs to register. By default registers only GPU 0.
)

cp.cuda.set_allocator(rmm_cupy_allocator)

In [ ]:

for condition in condition2model:
    model_path = [f'{work_dir}/model/{m}' for m in condition2model[condition]]
    for extra_str in ['', '_rb_5', '_rb_10', '_rb_20']:
        hits_all = []
        names_all = []
        for kind in ['footprint', 'count']:
            hits = scp.tl.seq_denovo_callhits(modisco_output=f'{work_dir}/modisco/modisco.{kind}.{condition}{extra_str}.h5',
                                       model_path=model_path,
                                       region_path=peak,
                                       device='cuda:0',
                                       preset=kind,
                                       save_path=f'{work_dir}/modisco/{kind}.{condition}{extra_str}/finemo',
                                       overwrite=False,
                                       verbose=True,
                                       launch=True, # Usually should be True when return_hits=True, but I run it once already
                                       return_hits=True)
            motif_uniq = np.sort(hits['motif_name'].unique())
            motif2id = {m:i for i,m in enumerate(motif_uniq)}
            ids = [motif2id[m] for m in hits['motif_name']]
            match_mm = np.zeros((len(peak_df), len(motif_uniq)))
            match_mm[hits['peak_id'], ids] += 1
            match_mm = (match_mm > 0)
            hits_all.append(match_mm)
            names_all += [f'{kind}.{condition}{extra_str}.{m}' for m in list(motif_uniq)]
        adata_atac = adata_atac_backup.copy()
        motif_matching = np.concatenate(hits_all, axis=-1)
        adata_atac.varm['motif_match'] = motif_matching
        adata_atac.uns['motif_name'] = names_all

        mm_df = pd.DataFrame(motif_matching, columns=names_all, index=adata_atac.var.index)
        mm_df.to_csv(os.path.join(work_dir, f'denovo_motifmatch{extra_str}.results.tsv.gz'), sep='\t', index=True, header=True)
        
        # cov = np.array(np.sum(adata_atac.X, axis=0)).reshape((-1))
        # adata_atac = adata_atac[:, cov > 0].copy()
        # scp.chromvar.sample_bg_peaks(adata_atac,
        #                      genome=scp.genome.hg38, 
        #                      method='chromvar', 
        #                      niterations=500)
        # chromvar_denovo = scp.chromvar.compute_deviations(adata_atac, 
        #                                           chunk_size=50000, 
        #                                           device='cuda')
        # chromvar_denovo.write(os.path.join(work_dir, f'denovochromvar{extra_str}.results.h5ad'))
        # chromvar_denovo_df = pd.DataFrame(chromvar_denovo.X, index=chromvar_denovo.obs.index, columns=chromvar_denovo.var.index)
        # chromvar_denovo_df.to_csv(os.path.join(work_dir, f'denovochromvar{extra_str}.results.tsv.gz'), sep='\t', index=True, header=True)

In [ ]:
print()

In [ ]:
mm_df.to_csv(os.path.join(work_dir, f'cisbp_motifmatch.results.tsv.gz'), sep='\t', index=True, header=True)
        

In [ ]:
import scprinter as scp
adata_atac = adata_atac_backup.copy()
cov = np.array(np.sum(adata_atac.X, axis=0)).reshape((-1))
adata_atac = adata_atac[:, cov > 0].copy()
scp.chromvar.sample_bg_peaks(adata_atac,
                             genome=scp.genome.hg38, 
                             method='chromvar', 
                             niterations=500)
motif = scp.motifs.FigR_Human_Motifs(scp.genome.hg38,
                                     bg=list(adata_atac.uns['bg_freq']), 
                                     n_jobs=100,
                                     pvalue=5e-5, mode='motifmatchr')
motif.prep_scanner(
    None,
    pvalue=5e-5,
)
motif.chromvar_scan(adata_atac)
mm_df = pd.DataFrame(adata_atac.varm['motif_match'], columns=adata_atac.uns['motif_name'], index=adata_atac.var.index)
mm_df.to_csv(os.path.join(work_dir, f'cisbp_motifmatch.results.tsv.gz'), sep='\t', index=True, header=True)
        

chromvar = scp.chromvar.compute_deviations(adata_atac, chunk_size=50000, device='cuda')
chromvar_denovo.write(os.path.join(work_dir, f'cisbpchromvar.results.h5ad'))
chromvar_denovo_df = pd.DataFrame(chromvar.X, index=chromvar.obs.index, columns=chromvar.var.index)
chromvar_denovo_df.to_csv(os.path.join(work_dir, f'cisbpchromvar.results.tsv.gz'), sep='\t', index=True, header=True)


In [ ]:
# fine-tunine
df = pd.read_csv(f'{work_dir}/pseudobulkids.louvains.txt', sep='\t')
cell_grouping, group_names = scp.utils.df2cell_grouping(printer, df)

In [ ]:
embeddings = pd.read_csv(f'{work_dir}/cell_100topic_embedding.txt', sep='\t')

In [ ]:
embeddings.index = embeddings['cell']

In [ ]:
embeddings = embeddings.iloc[:, 1:]
embeddings

In [ ]:
pretrain_models = condition2model['Bulk']
pretrain_models.sort()
pretrain_models

In [ ]:
lora_configs = []
for fold, model in enumerate(pretrain_models):
    lora_config = scp.tl.seq_lora_model_config(printer,
                                            region_path=f'{work_dir}/seq2print_Bulk_cleaned_narrowPeak.bed',
                                            cell_grouping=cell_grouping,
                                            group_names=group_names,
                                            additional_group_attr=None,
                                            embeddings=embeddings,
                                            genome=scp.genome.hg38,
                                            pretrain_model=f'{work_dir}/model/{model}',
                                            overwrite_barcode=True,
                                            model_name=f'MORF_20_leiden',
                                            fold=fold,
                                            model_config=f'{work_dir}/configs/Bulk_fold{fold}.JSON',
                                            additional_lora_config={
                                            "notes": "v3",
                                            "tags": ["MORF",
                                                     "Bulk",
                                                     "Leiden20"
                                                     "LoRA",
                                                f"fold{fold}"]},
                                            path_swap=(work_dir, ''),
                                            config_save_path=f'{work_dir}/configs/MORF_leiden20_lora_fold{fold}.JSON')
    lora_configs.append(lora_config)

In [ ]:
name = 'Bulk'
for fold in range(5):
    scp.tl.launch_seq2print(model_config_path=f'{work_dir}/configs/MORF_leiden20_lora_fold{fold}.JSON',
                            temp_dir=f'{work_dir}/temp',
                            model_dir=f'{work_dir}/model',
                            data_dir=f'{work_dir}',
                            gpus=fold,
                            wandb_project='scPrinter_seq_RSav_morf', # wandb helps you manage loggins
                            verbose=False, 
                            launch=False # launch=True, this command would launch the scripts directly, 
                            # otherwise, it will just display the commands, you should copy them and run them.
                           )

In [ ]:
peak = os.path.join(work_dir, 'cleanpeaks.bed')

In [ ]:
# Now fetch models:

import wandb

# Login to your W&B account
wandb.login()

# Set your entity and project
entity = 'ruochiz'  # Replace with your W&B entity (username or team name)
project = 'scPrinter_seq_RSav_morf'  # Replace with your W&B project name

# Initialize the API
api = wandb.Api()

# Get the project
runs = api.runs(f"{entity}/{project}")
lora_models = []
for run in runs:
    model_name = run.config['savename'] + '-' + run.name + '.pt'
    if run.state == 'finished':
        for tag in run.tags:
            if "Leiden20" not in tag:
                continue
            if 'fold' in tag:
                continue
            lora_models.append(model_name)

In [ ]:
lora_models.sort()

In [ ]:
fold = 0
scp.tl.seq_tfbs_seq2print(
    seq_attr_count=None,
    seq_attr_footprint=None,
    genome=printer.genome,
    region_path=peak,
    gpus=[3,4,2],
    model_type="lora",
    model_path=[f'{work_dir}/model/{model}' for model in lora_models],
    lora_config=f'{work_dir}/configs/MORF_leiden20_lora_fold{fold}.JSON',
    group_names=[[x] for x in group_names],
    save_group_names=group_names,
    save_path=f'{work_dir}/tfbs_leiden20/',
    overwrite_seqattr=False,
    post_normalize=True,
    verbose= False,
    launch=False,
    return_adata=False,
    save_key='cleanpeaks',
)

In [ ]:
fold = 0
adata = scp.tl.seq_tfbs_seq2print(
    seq_attr_count=None,
    seq_attr_footprint=None,
    genome=printer.genome,
    region_path=os.path.join(work_dir, 'toxpeaks.bed'),
    gpus=[0],
    model_type="lora",
    model_path=[f'{work_dir}/model/{model}' for model in lora_models],
    lora_config=f'{work_dir}/configs/MORF_leiden20_lora_fold{fold}.JSON',
    group_names=[[x] for x in group_names],
    save_group_names=group_names,
    save_path=f'{work_dir}/tfbs_leiden20_test_region/',
    overwrite_seqattr=False,
    post_normalize=True,
    verbose= False,
    launch=True,
    return_adata=True,
    save_key='toxpeaks',
)

In [ ]:
adata_post_normalize = adata.copy()

In [ ]:
fold = 0
adata = scp.tl.seq_tfbs_seq2print(
    seq_attr_count=None,
    seq_attr_footprint=None,
    genome=printer.genome,
    region_path=os.path.join(work_dir, 'toxpeaks.bed'),
    gpus=[0],
    model_type="lora",
    model_path=[f'{work_dir}/model/{model}' for model in lora_models],
    lora_config=f'{work_dir}/configs/MORF_leiden20_lora_fold{fold}.JSON',
    group_names=[[x] for x in group_names],
    save_group_names=group_names,
    save_path=f'{work_dir}/tfbs_leiden20_test_region/',
    overwrite_seqattr=False,
    post_normalize=False,
    verbose= False,
    launch=True,
    return_adata=True,
    save_key='toxpeaks',
)


In [ ]:
adata_post_normalize.obs['group_name'] = [str(xx) for xx in adata_post_normalize.obs['group_name']]

In [ ]:
adata.obs['group_name'] = [str(xx) for xx in adata.obs['group_name']]

In [ ]:
adata_post_normalize.write(f'{work_dir}/tfbs_leiden20_test_region/leiden20_tox_post_normalize_tfbs.h5ad')
adata.write(f'{work_dir}/tfbs_leiden20_test_region/leiden20_tox_tfbs.h5ad')

In [ ]:
for group in trange(0, 22):
    hypo_path = f"/ewsc/zhangruo/RSav_scprinter/model/broad_louvain_{group}.footprint.avg.hypo.npz"
    hypo = []
    for model in lora_models:
        hypo.append(f'/ewsc/zhangruo/RSav_scprinter/model/{model}_cleanpeaks/model_broad_louvain_{group}.hypo.just_sum.shap_hypo_0-30_.0.85.npz')
    new_hypo = [np.load(h)["arr_0"] for h in hypo]
    new_hypo = np.array(new_hypo).mean(axis=0)
    np.savez(hypo_path, new_hypo)
for group in trange(0, 22):
    hypo_path = f"/ewsc/zhangruo/RSav_scprinter/model/broad_louvain_{group}.count.avg.hypo.npz"
    hypo = []
    for model in lora_models:
        hypo.append(f'/ewsc/zhangruo/RSav_scprinter/model/{model}_cleanpeaks/model_broad_louvain_{group}.hypo.count.shap_hypo_0_.0.85.npz')
    new_hypo = [np.load(h)["arr_0"] for h in hypo]
    new_hypo = np.array(new_hypo).mean(axis=0)
    np.savez(hypo_path, new_hypo)

In [ ]:
printer.close()